# QProgram 101 — The Building Blocks

Welcome! This notebook is a guided tour of the core concepts in `qprogram`, the same vocabulary the [QProgram DSL spec](https://www.notion.so/qilimanjaro/QProgram-DSL-Specification-Draft-32f7eec14c53815a8290d85478cdcaec) covers, but in a hands-on, code-first style.

By the end you'll know how to:

- assemble a `QProgram` from operations, variables, and control flow
- reference hardware buses three different ways (plain strings, preset schemas, custom schemas)
- build waveforms and sweep their parameters
- save a program to a `.qp` file and reload it byte-stably
- read results that come back from execution

In [ ]:
import numpy as np

import qprogram as qp
from qprogram.buses import BusSchema, BusNaming
from qprogram.waveforms import Square, Gaussian, IQPair, IQDrag

## 1. The `QProgram` container

A `QProgram` is a tree of operations and control-flow blocks plus some metadata. It's a *description* of a pulse sequence — not yet bound to any hardware. A platform compiles it and runs it; the program itself stays portable.

In [ ]:
program = qp.QProgram(label="hello", description="My first QProgram")

# Read-only views of what's inside
print('label       :', program.label)
print('description :', program.description)
print('body        :', program.body)            # the root Block (empty for now)
print('variables   :', program.variables)       # no variables declared yet
print('buses       :', program.buses)           # no buses referenced yet

label       : hello
description : My first QProgram
body        : <qprogram.blocks.block.Block object at 0x715e102048c0>
variables   : []
buses       : set()


## 2. Operations — the leaves of the tree

Operations are what actually run on the hardware. Most target a **bus** (a hardware channel: drive line, readout line, flux line, etc.) and do something on it. The most common ones:

| Operation | What it does |
|---|---|
| `play(bus, waveform)` | Output a waveform |
| `measure(bus, waveform, weights)` | Play a readout pulse and acquire the response |
| `wait(bus, duration)` | Idle for `duration` ns |
| `sync(buses=None)` | Align timing across buses (all buses if `None`) |
| `set_frequency(bus, frequency)` | Set the NCO frequency on a bus |
| `set_phase`, `reset_phase`, `set_gain`, `set_offset` | Other real-time parameter controls |

Let's add a few operations and see the program grow:

In [ ]:
square = Square(amplitude=0.5, duration=100)
iq_drag = IQDrag(amplitude=0.5, duration=40, num_sigmas=2.5, drag_coefficient=0.1)

program = qp.QProgram(label='demo')
program.play("drive_q0", square)
program.wait("drive_q0", 200)
program.sync()

print('buses now referenced:', sorted(program.buses))
print('body has', len(program.body.elements), 'operations')

buses now referenced: ['drive_q0']
body has 3 operations


## 3. Buses — three ways to reference them

Every operation targets a bus *by name*. There are three ways the user can produce that name, each with different trade-offs:

1. **Plain strings** — fastest to type, no validation, no autocomplete.
2. **Preset schemas** — typed accessors for common qubit topologies (transmon, fluxonium, etc.). Full IDE autocomplete and validation.
3. **Custom schemas** — define your own elements and bus kinds for exotic topologies.

All three end up as strings at the AST level — the differences are in *how the user spells them* and in what metadata travels along.

### 3.1 Plain strings

The simplest form. Just pass a string. No validation, no metadata; you're on your own for typos.

In [ ]:
p = qp.QProgram()
p.play("drive_q0_bus", square)
p.measure("readout_q0_bus", iq_drag, iq_drag)
print(sorted(p.buses))

['drive_q0_bus', 'readout_q0_bus']


### 3.2 Preset schemas

`BusSchema` ships with presets for common qubit kinds. You get typed accessors (`q[0].drive`, `q[0].readout`) and runtime validation — e.g. trying to `measure()` on a non-acquiring bus raises `TypeError` at program-construction time.

Available presets:

- `BusSchema.transmon()` — `.q` with `drive` (IQ) and `readout` (IQ, acquires).
- `BusSchema.transmon_coupled()` — adds `.c` with `flux` (single).
- `BusSchema.flux_tunable_transmon[_coupled]()` — adds `flux` (single) to `.q`.
- `BusSchema.fluxonium[_coupled]()` — `.q` with `drive`, `readout`, `flux_x`, `flux_z`.

Each `BusRef` (e.g. `q[0].drive`) is a `str` subclass that *also* carries `element`, `index`, `kind`, `channel`, `acquires`, and a back-reference to the `schema`. So it works wherever a string works, but the validators (and downstream tooling) see the metadata.

In [ ]:
schema = BusSchema.flux_tunable_transmon()
q = schema.q

drive = q[0].drive
print('value       :', repr(str(drive)))
print('element     :', drive.element)
print('index       :', drive.index)
print('kind        :', drive.kind)
print('channel     :', drive.channel)
print('acquires    :', drive.acquires)
print('schema.name :', drive.schema.name)

value       : 'q0/drive'
element     : q
index       : 0
kind        : drive
channel     : IQ
acquires    : False
schema.name : flux_tunable_transmon


In [ ]:
# Validation kicks in automatically when buses come from a schema.
p = qp.QProgram()
p.play(q[0].drive, IQDrag(amplitude=0.5, duration=40, num_sigmas=2.5, drag_coefficient=0.1))  # OK: IQ waveform on IQ bus
try:
    p.play(q[0].flux, IQDrag(amplitude=0.5, duration=40, num_sigmas=2.5, drag_coefficient=0.1))            # single waveform on IQ bus → TypeError
except TypeError as e:
    print('caught:', e)
try:
    p.measure(q[0].drive, iq_drag, iq_drag)              # drive has no ADC → TypeError
except TypeError as e:
    print('caught:', e)

caught: Bus 'q0/flux' is a single channel but received an IQWaveform (IQDrag). Use a single-channel Waveform (e.g. Square, FlatTop) instead.
caught: Bus 'q0/drive' does not support acquisition (acquires=False). measure() can only be called on buses with an ADC (e.g. readout buses).


**Custom naming pattern.** If your platform uses a different convention than the default `"{element}{index}/{kind}"`, pass a `BusNaming(pattern=...)`. The pattern supports `{element}`, `{index}`, `{kind}`:

In [ ]:
qbox = BusSchema.transmon(naming=BusNaming("{kind}_{element}{index}_bus"))
print(qbox.q[0].drive)        # 'drive_q0_bus'
print(qbox.q[0].readout)      # 'readout_q0_bus'

drive_q0_bus
readout_q0_bus


### 3.3 Custom (dynamic) schemas

For topologies the presets don't cover — extra bus kinds, non-standard elements, exotic chips — build your own with `BusSchema(name=...)` and `add_element`. The user-facing trade-off is that dynamic schemas lose static typing (you access `.q[0].drive` via `__getattr__` instead of an `@property`), but everything else works the same: validation, serialization, all of it.

Each element maps a bus kind name to a `(channel, acquires)` tuple.

In [ ]:
chip = BusSchema(name="chip")
chip.add_element("q", buses={
    "drive":   ("IQ", False),
    "readout": ("IQ", True),
    "flux":    ("single", False),
})
chip.add_element("c", buses={"flux": ("single", False)})

# Access works the same as presets, just without IDE autocomplete on `.q`, `.c`
print(chip.q[0].drive, '   ', chip.q[0].drive.channel, chip.q[0].drive.acquires)
print(chip.c[0,1].flux, '   ', chip.c[0,1].flux.channel)

q0/drive     IQ False
c0_1/flux     single


## 4. Variables

Variables let you parameterize a program. They appear in loops (the loop variable steps through values) and inside expressions (waveform amplitudes, frequencies, durations, etc.).

Each variable has:

- **`id`** — mandatory. Must match `[A-Za-z_][A-Za-z0-9_]*` — letters/digits/underscores only. Unique within a `QProgram`. This is the identifier used in `.qp` files.
- **`label`** — optional human-readable name (axis labels, plot titles, result-array dimensions).
- **`units`** — optional unit string (`"Hz"`, `"ns"`, `"V"`).
- **`description`** — optional longer description.

Only `id` flows into the runtime; the rest is metadata for tooling and results.

In [ ]:
p = qp.QProgram()

freq = p.variable(
    "freq",                                 # id
    label="Drive frequency",                   # human-readable
    units="Hz",
    description="NCO carrier swept across the qubit transition",
)

print(freq.id, '|', freq.label, '|', freq.units)

freq | Drive frequency | Hz


### Expressions

Variables compose with arithmetic operators (`+`, `-`, `*`, `/`, unary `-`/`+`). Anywhere a numeric parameter is accepted you can pass a literal, a variable, or any expression built from them. The runtime executor assigns values per loop iteration; expressions evaluate naturally.

In [ ]:
t = p.variable("t")
amp = p.variable("amp")

# Build expressions on the fly
delay = 100 + t
amplitude_envelope = amp / 2
frequency_offset = freq + t * 1e6

# Drop them into operations — they're held symbolically until execution
p.wait("drive_q0", 100 + t)
p.set_frequency("drive_q0", freq + t * 1e6)
p.set_gain("drive_q0", amp / 2)
print('len(program.body):', len(p.body.elements))

len(program.body): 3


## 5. Waveforms

Waveforms describe pulse shapes. They're pure data objects — no hardware command. There are two families: **single-channel** (`Waveform`) and **IQ** (`IQWaveform`, two-channel complex). The schema validators in §3.2 reject the wrong family.

Common built-ins:

- **Single-channel**: `Square`, `Gaussian`, `GaussianDragCorrection`, `Ramp`, `FlatTop`, `SuddenNetZero`, `Arbitrary`, `Chained`.
- **IQ**: `IQPair(I, Q)` pairs any two single-channel waveforms; `IQDrag` is a ready-made DRAG pulse.

All parameters accept variables, so you can sweep waveform shape from inside a loop.

In [ ]:
# Plain shapes
sq = Square(amplitude=0.5, duration=100)
gauss = Gaussian(amplitude=0.5, duration=40, num_sigmas=2.5)

# IQ pulses
iq_pulse = IQPair(I=Square(0.5, 100), Q=Square(0.0, 100))
drag = IQDrag(amplitude=0.5, duration=40, num_sigmas=2.5, drag_coefficient=0.1)

# Variable-aware: amplitude and duration can be variables, swept by an enclosing loop
p = qp.QProgram()
amp = p.variable("amp")
swept = Gaussian(amplitude=amp, duration=40, num_sigmas=2.5)
print(type(swept).__name__, '— amplitude is', type(swept.amplitude).__name__)

Gaussian — amplitude is Variable


## 6. Control flow

Control flow is expressed via `with`-blocks that build nested AST nodes:

- **`for_loop(variable, start, stop, step=1)`** — parametric sweep.
- **`loop(variable, values)`** — sweep over an arbitrary numpy array.
- **`average(shots)`** — repeat & average the inner block.
- **`block()`** — a generic grouping scope.
- **Parallel loops** — combine loops with the `|` operator: `with for_loop(a, ...) | for_loop(b, ...):` runs them concurrently.

Crucially, the program makes no distinction between *hardware* and *software* loops. You describe the sweep; the compiler decides how to run it.

In [ ]:
p = qp.QProgram()
freq = p.variable("freq", units="Hz")
gain = p.variable("gain")

with p.average(shots=1000):
    with p.for_loop(freq, start=4.5e9, stop=5.5e9, step=1e6):
        with p.for_loop(gain, start=0.0, stop=1.0, step=0.05):
            p.set_frequency("drive_q0", freq)
            p.set_gain("drive_q0", gain)
            p.play("drive_q0", gauss)
            p.sync()
            p.measure("readout_q0", iq_pulse, iq_pulse)

print('body depth:', len(p.body.elements), 'top-level element')

body depth: 1 top-level element


### Parallel loops

Sweep two variables in lock-step (same number of iterations, advancing together):

In [ ]:
p = qp.QProgram()
a = p.variable("a")
b = p.variable("b")

with p.for_loop(a, 0.0, 1.0, 0.01) | p.for_loop(b, 0, 100, 1):
    p.set_gain("drive_q0", a)
    p.wait("drive_q0", b)

# The two loops share an iteration axis; results coming back will have a single combined dim.

## 7. Save / load: the `.qp` format

`qp.dumps(program)` / `qp.loads(text)` (and `qp.save` / `qp.load` for files) round-trip a `QProgram` through a small text format. Three things to know:

- **Plain string buses** stay quoted: `play "drive_q0" pulse`.
- **Preset schemas** emit a one-liner: `schema: transmon` (optionally aliased or with a custom naming pattern). Operations reference them as paths: `play transmon.q[0].drive pulse`.
- **Custom schemas** emit an inline block with their elements and bus kinds.

Round-trip is byte-stable: `dumps(loads(text)) == text`.

In [ ]:
# A program mixing all three cases
preset = BusSchema.transmon()
chip = BusSchema(name="chip")
chip.add_element("q", {"drive": ("IQ", False), "readout": ("IQ", True), "flux": ("single", False)})

p = qp.QProgram(label="mixed")
freq = p.variable("freq", label="Drive frequency", units="Hz")
with p.for_loop(freq, 4e9, 6e9, 1e6):
    p.set_frequency(preset.q[0].drive, freq)              # preset schema
    p.measure(preset.q[0].readout, iq_pulse, iq_pulse)    # preset schema
    p.set_offset(chip.q[2].flux, 0.5)                     # custom schema
    p.play("raw_bus", sq)                                 # plain string

text = qp.dumps(p)
print(text)

#!QProgram 1.0

metadata:
  label: "mixed"

schema chip:
  element q:
    drive info=IQ
    readout info=IQ+acquires
    flux info=single

schema: transmon

body:
  var freq label="Drive frequency" units="Hz"

  for freq in range(4000000000.0, 6000000000.0, 1000000.0):
    set_frequency transmon.q[0].drive freq
    measure transmon.q[0].readout IQPair(I=Square(amplitude=0.5, duration=100), Q=Square(amplitude=0.0, duration=100)) IQPair(I=Square(amplitude=0.5, duration=100), Q=Square(amplitude=0.0, duration=100))
    set_offset chip.q[2].flux 0.5
    play "raw_bus" Square(amplitude=0.5, duration=100)



In [ ]:
# Round-trip — load it back and confirm byte-stability
reloaded = qp.loads(text)
assert qp.dumps(reloaded) == text
print('round-trip OK; reloaded buses:', sorted(reloaded.buses))

round-trip OK; reloaded buses: ['q0/drive', 'q0/readout', 'q2/flux', 'raw_bus']


## 8. Results — a quick look

Executing a `QProgram` returns a `QProgramResult`. Each `measure()` call produces an `xarray.DataArray` with one dimension per enclosing loop plus a final `IQ` dimension (`I`, `Q`). You can `.sel()` by loop variable, slice, and convert to numpy/pandas as usual.

We won't execute a real program here — that needs a platform — but the API looks like this:

```python
result = platform.execute(program)

da = result.get(measurement=0)            # xarray.DataArray
da.dims                                    # ('freq', 'gain', 'IQ')
I = da.sel(IQ='I')
Q = da.sel(IQ='Q')
complex_s21 = I + 1j * Q
```

The `spectroscopy_example.ipynb` notebook walks through a complete result analysis using simulated data.

## What's next

You've seen all the moving parts:

- `QProgram` as a tree of operations and control-flow blocks
- Three flavours of bus references (plain / preset / custom)
- Variables, expressions, and waveforms
- Control flow with `average`, `for_loop`, `loop`, and parallel via `|`
- `.qp` save/load round-trip
- The shape of results